# PyTorch Starter Templates
Runnable binary and multi-class classification templates following recommended patterns from the cheat sheet:
- **Raw logits** out of the model (no final activation)
- Let the **loss function** handle sigmoid/softmax internally
- Apply activation **manually at evaluation time**

In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---
## Binary Classification

In [100]:
torch.manual_seed(42)

# ================================
# --> Data
# ================================
N_SAMPLES = 256
N_FEATURES = 10

x = torch.randn(N_SAMPLES, N_FEATURES, device=device)
y = torch.randint(0, 2, (N_SAMPLES, 1), device=device).float()

# ================================
# --> Model (raw logits, no sigmoid)
# ================================
model = nn.Sequential(
    nn.Linear(N_FEATURES, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
).to(device)

# ================================
# --> Loss & optimizer
# ================================
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ================================
# --> Training loop
# ================================
for step in range(500):
    logits = model(x)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step:>4d} | loss: {loss.item():.4f}")

# ================================
# --> Evaluation
# ================================
with torch.no_grad():
    probs = torch.sigmoid(model(x))      # apply sigmoid to logits
    preds = (probs > 0.5).float()         # threshold at 0.5
    accuracy = (preds == y).float().mean()
    print(f"\nAccuracy: {accuracy.item():.2%}")

# ================================
# --> Parameter count
# ================================
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters - total: {total:,}  trainable: {trainable:,}")

for name, p in model.named_parameters():
    print(f"  {name:>12s}  {str(list(p.shape)):>12s}  {p.numel():>6,}")

step    0 | loss: 0.6933
step  100 | loss: 0.6107
step  200 | loss: 0.4261
step  300 | loss: 0.2556
step  400 | loss: 0.1261

Accuracy: 100.00%
Parameters - total: 897  trainable: 897
      0.weight      [32, 10]     320
        0.bias          [32]      32
      2.weight      [16, 32]     512
        2.bias          [16]      16
      4.weight       [1, 16]      16
        4.bias           [1]       1


---
## Multi-Class Classification

In [101]:
torch.manual_seed(42)

# ================================
# --> Data
# ================================
N_SAMPLES = 256
N_FEATURES = 10
N_CLASSES = 4

x = torch.randn(N_SAMPLES, N_FEATURES, device=device)
y = torch.randint(0, N_CLASSES, (N_SAMPLES,), device=device)  # class indices, NOT one-hot

# ================================
# --> Model (raw logits, no softmax)
# ================================
model = nn.Sequential(
    nn.Linear(N_FEATURES, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, N_CLASSES),
).to(device)

# ================================
# --> Loss & optimizer
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ================================
# --> Training loop
# ================================
for step in range(5000):
    logits = model(x)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step:>4d} | loss: {loss.item():.4f}")

# ================================
# --> Evaluation
# ================================
with torch.no_grad():
    logits = model(x)
    preds = logits.argmax(dim=-1)         # argmax on logits
    accuracy = (preds == y).float().mean()
    print(f"\nAccuracy: {accuracy.item():.2%}")

# ================================
# --> Parameter count
# ================================
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters - total: {total:,}  trainable: {trainable:,}")

for name, p in model.named_parameters():
    print(f"  {name:>12s}  {str(list(p.shape)):>12s}  {p.numel():>6,}")

step    0 | loss: 1.3765
step  100 | loss: 1.2077
step  200 | loss: 0.9387
step  300 | loss: 0.6442
step  400 | loss: 0.3934
step  500 | loss: 0.2157
step  600 | loss: 0.1152
step  700 | loss: 0.0642
step  800 | loss: 0.0390
step  900 | loss: 0.0257
step 1000 | loss: 0.0180
step 1100 | loss: 0.0133
step 1200 | loss: 0.0100
step 1300 | loss: 0.0078
step 1400 | loss: 0.0062
step 1500 | loss: 0.0051
step 1600 | loss: 0.0042
step 1700 | loss: 0.0036
step 1800 | loss: 0.0030
step 1900 | loss: 0.0026
step 2000 | loss: 0.0023
step 2100 | loss: 0.0020
step 2200 | loss: 0.0017
step 2300 | loss: 0.0015
step 2400 | loss: 0.0014
step 2500 | loss: 0.0012
step 2600 | loss: 0.0011
step 2700 | loss: 0.0010
step 2800 | loss: 0.0009
step 2900 | loss: 0.0008
step 3000 | loss: 0.0007
step 3100 | loss: 0.0007
step 3200 | loss: 0.0006
step 3300 | loss: 0.0006
step 3400 | loss: 0.0005
step 3500 | loss: 0.0005
step 3600 | loss: 0.0004
step 3700 | loss: 0.0004
step 3800 | loss: 0.0004
step 3900 | loss: 0.0003
